In [2]:
import nest_asyncio

nest_asyncio.apply()

import os

os.environ["OPENAI_API_KEY"] = ""

In [4]:
from llama_index.core import SimpleDirectoryReader

documents = SimpleDirectoryReader("./data/paul_graham/").load_data()

In [5]:
from llama_index.core import Document
from llama_index.embeddings.openai import OpenAIEmbedding
from llama_index.core.node_parser import TokenTextSplitter
from llama_index.core.extractors import TitleExtractor
from llama_index.core.ingestion import IngestionPipeline, IngestionCache

In [6]:
pipeline = IngestionPipeline(
    transformations=[
        TokenTextSplitter(chunk_size=1024, chunk_overlap=100),
    ]
)
nodes = pipeline.run(documents=documents)

In [9]:
nodes[0]

TextNode(id_='892bee29-fa48-46e0-a5b2-3476b11880d2', embedding=None, metadata={'file_path': 'D:\\work\\AI\\kb-management\\data\\paul_graham\\paul_graham_essay.txt', 'file_name': 'paul_graham_essay.txt', 'file_type': 'text/plain', 'file_size': 75041, 'creation_date': '2025-06-04', 'last_modified_date': '2025-06-04'}, excluded_embed_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], excluded_llm_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], relationships={<NodeRelationship.SOURCE: '1'>: RelatedNodeInfo(node_id='6b9cffb9-62f1-4b33-9e00-ea6f4d971780', node_type=<ObjectType.DOCUMENT: '4'>, metadata={'file_path': 'D:\\work\\AI\\kb-management\\data\\paul_graham\\paul_graham_essay.txt', 'file_name': 'paul_graham_essay.txt', 'file_type': 'text/plain', 'file_size': 75041, 'creation_date': '2025-06-04', 'last_modified_date': '2025-06-04'}, hash='50fffb84c013df2e92

In [10]:
pipeline = IngestionPipeline(
    transformations=[
        TokenTextSplitter(chunk_size=1024, chunk_overlap=100),
        TitleExtractor(),
    ]
)
nodes = pipeline.run(documents=documents)

100%|███████████████████████████████████████████████████████████████████████| 5/5 [00:05<00:00,  1.04s/it]


In [12]:
nodes[0].metadata["document_title"]

'From Writing and Art to Programming: A Journey through AI, Computer Science, and Software Development'

In [13]:
pipeline = IngestionPipeline(
    transformations=[
        TokenTextSplitter(chunk_size=1024, chunk_overlap=100),
        TitleExtractor(),
        OpenAIEmbedding(),
    ]
)
nodes = pipeline.run(documents=documents)

100%|███████████████████████████████████████████████████████████████████████| 5/5 [00:01<00:00,  3.06it/s]


In [14]:
nodes[0].metadata["document_title"]

'From Writing to Programming: Navigating the Intersection of Art, Academia, and Business Strategy in AI and Painting'

In [16]:
pipeline = IngestionPipeline(
    transformations=[
        TokenTextSplitter(chunk_size=1024, chunk_overlap=100),
        TitleExtractor(),
    ]
)
nodes = pipeline.run(documents=documents)

100%|███████████████████████████████████████████████████████████████████████| 5/5 [00:01<00:00,  3.01it/s]


In [18]:
# save and load
pipeline.cache.persist("./llama_cache.json")
new_cache = IngestionCache.from_persist_path("./llama_cache.json")

In [19]:
new_pipeline = IngestionPipeline(
    transformations=[
        TokenTextSplitter(chunk_size=1024, chunk_overlap=100),
        TitleExtractor(),
    ],
    cache=new_cache,
)

In [20]:
nodes = new_pipeline.run(documents=documents)

In [21]:
pipeline = IngestionPipeline(
    transformations=[
        TokenTextSplitter(chunk_size=1024, chunk_overlap=100),
        TitleExtractor(),
        OpenAIEmbedding(),
    ],
    cache=new_cache,
)
nodes = pipeline.run(documents=documents)

In [24]:
# save and load
pipeline.cache.persist("./nodes_embedding.json")
nodes_embedding_cache = IngestionCache.from_persist_path(
    "./nodes_embedding.json"
)

In [23]:
pipeline = IngestionPipeline(
    transformations=[
        TokenTextSplitter(chunk_size=1024, chunk_overlap=100),
        TitleExtractor(),
        OpenAIEmbedding(),
    ],
    cache=nodes_embedding_cache,
)

# Will load it from the cache as the transformations are same.
nodes = pipeline.run(documents=documents)

In [ ]:
import nest_asyncio

nest_asyncio.apply()

import os

os.environ["OPENAI_API_KEY"] = ""

from llama_index.core import SimpleDirectoryReader

documents = SimpleDirectoryReader("./data/paul_graham/").load_data()

from llama_index.core import Document
from llama_index.embeddings.openai import OpenAIEmbedding
from llama_index.core.node_parser import TokenTextSplitter
from llama_index.core.extractors import TitleExtractor
from llama_index.core.ingestion import IngestionPipeline, IngestionCache
from llama_index.vector_stores.pinecone import PineconeVectorStore

from pinecone import Pinecone, ServerlessSpec

pc = Pinecone(api_key="")

index_name = "llama-integration-example"

# pc.create_index(
#     index_name,
#     dimension=1536,
#     spec=ServerlessSpec(cloud="aws", region="us-east-1"),
# )

pinecone_index = pc.Index(index_name)

vector_store = PineconeVectorStore(pinecone_index=pinecone_index)

nodes_embedding_cache = IngestionCache.from_persist_path(
    "./nodes_embedding.json"
)

pipeline = IngestionPipeline(
    transformations=[
        TokenTextSplitter(chunk_size=1024, chunk_overlap=100),
        TitleExtractor(),
        OpenAIEmbedding(),
    ],
    vector_store=vector_store,
    cache=nodes_embedding_cache, 
)

# Will load it from the cache as the transformations are same.
pipeline.run(documents=documents)

print(pinecone_index.describe_index_stats())


In [1]:
import nest_asyncio

nest_asyncio.apply()

import os

os.environ["OPENAI_API_KEY"] = ""

from llama_index.core import SimpleDirectoryReader

documents = SimpleDirectoryReader("./data/paul_graham/").load_data()

from llama_index.core import Document
from llama_index.embeddings.openai import OpenAIEmbedding
from llama_index.core.node_parser import TokenTextSplitter
from llama_index.core.extractors import TitleExtractor
from llama_index.core.ingestion import IngestionPipeline, IngestionCache
from llama_index.vector_stores.pinecone import PineconeVectorStore

from pinecone import Pinecone, ServerlessSpec

pc = Pinecone(api_key="")


In [2]:
from llama_index.core import VectorStoreIndex
from llama_index.core.retrievers import VectorIndexRetriever
index_name = "llama-integration-example"

pinecone_index = pc.Index(index_name)

vector_store = PineconeVectorStore(pinecone_index=pinecone_index)
# Instantiate VectorStoreIndex object from your vector_store object
vector_index = VectorStoreIndex.from_vector_store(vector_store=vector_store)

# Grab 5 search results
retriever = VectorIndexRetriever(index=vector_index, similarity_top_k=5)

In [6]:
answer = retriever.retrieve('Pauls way of life')

# Inspect results
print([i.get_content() for i in answer])

['goal, or it would have been hard to keep at it for so long.\n\nI wrote this new Lisp, called Bel, in itself in Arc. That may sound like a contradiction, but it\'s an indication of the sort of trickery I had to engage in to make this work. By means of an egregious collection of hacks I managed to make something close enough to an interpreter written in itself that could actually run. Not fast, but fast enough to test.\n\nI had to ban myself from writing essays during most of this time, or I\'d never have finished. In late 2015 I spent 3 months writing essays, and when I went back to working on Bel I could barely understand the code. Not so much because it was badly written as because the problem is so convoluted. When you\'re working on an interpreter written in itself, it\'s hard to keep track of what\'s happening at what level, and errors can be practically encrypted by the time you get them.\n\nSo I said no more essays till Bel was done. But I told few people about Bel while I was 

In [2]:
!pip install llama-cloud-services

  Using cached llama_cloud-0.1.23-py3-none-any.whl.metadata (1.1 kB)
  Using cached llama_index_core-0.12.40-py3-none-any.whl.metadata (2.4 kB)
  Using cached platformdirs-4.3.8-py3-none-any.whl.metadata (12 kB)
  Using cached certifi-2025.4.26-py3-none-any.whl.metadata (2.5 kB)
  Using cached banks-2.1.2-py3-none-any.whl.metadata (12 kB)
  Using cached Deprecated-1.2.18-py2.py3-none-any.whl.metadata (5.7 kB)
  Using cached dirtyjson-1.0.8-py3-none-any.whl.metadata (11 kB)
  Using cached filetype-1.2.0-py2.py3-none-any.whl.metadata (6.5 kB)
  Using cached fsspec-2025.5.1-py3-none-any.whl.metadata (11 kB)
  Using cached networkx-3.5-py3-none-any.whl.metadata (6.3 kB)
  Using cached nltk-3.9.1-py3-none-any.whl.metadata (2.9 kB)
  Using cached pillow-11.2.1-cp312-cp312-win_amd64.whl.metadata (9.1 kB)
  Using cached tqdm-4.67.1-py3-none-any.whl.metadata (57 kB)
  Using cached wrapt-1.17.2-cp312-cp312-win_amd64.whl.metadata (6.5 kB)
  Using cached griffe-1.7.3-py3-none-any.whl.metadata (5.0

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langgraph-api 0.0.15 requires langgraph<0.3.0,>=0.2.56, but you have langgraph 0.2.45 which is incompatible.


In [4]:
from llama_cloud_services import LlamaParse

parser = LlamaParse(api_key="")
result = await parser.aparse("C:\\Users\namis\Downloads\budget_speech.pdf")

Started parsing the file under job_id 6252c369-5ea8-4609-8352-0830d5778b5c


In [5]:
result

JobResult(pages=[Page(page=1, text='     GOVERNMENT OF INDIA\n  BUDGET 2025-2026\n       SPEECH\n         OF\n NIRMALA SITHARAMAN\nMINISTER OF FINANCE\n     February 1, 2025', md='# GOVERNMENT OF INDIA\n\n# BUDGET 2025-2026\n\n# SPEECH OF NIRMALA SITHARAMAN\n\n# MINISTER OF FINANCE\n\nFebruary 1, 2025', images=[], charts=[], tables=[], layout=[], items=[PageItem(type='heading', lvl=1, value='GOVERNMENT OF INDIA', md='# GOVERNMENT OF INDIA', rows=None, bBox=BBox(x=221.0, y=228.92, w=117.0, h=276.16)), PageItem(type='heading', lvl=1, value='BUDGET 2025-2026', md='# BUDGET 2025-2026', rows=None, bBox=BBox(x=192.0, y=341.0, w=172.0, h=347.08)), PageItem(type='heading', lvl=1, value='SPEECH OF NIRMALA SITHARAMAN', md='# SPEECH OF NIRMALA SITHARAMAN', rows=None, bBox=BBox(x=157.0, y=467.08, w=241.0, h=64.0)), PageItem(type='heading', lvl=1, value='MINISTER OF FINANCE', md='# MINISTER OF FINANCE', rows=None, bBox=BBox(x=208.0, y=491.08, w=140.0, h=56.92)), PageItem(type='text', lvl=None, valu

In [6]:
result.job_metadata

JobMetadata(job_pages=60, job_auto_mode_triggered_pages=0, job_is_cache_hit=False)

In [11]:
result.get_markdown_documents()[0].text

'# GOVERNMENT OF INDIA\n\n# BUDGET 2025-2026\n\n# SPEECH OF NIRMALA SITHARAMAN\n\n# MINISTER OF FINANCE\n\nFebruary 1, 2025\n---\nNO_CONTENT_HERE\n---\n# CONTENTS\n\n# PART – A\n\n| Introduction                  | 1  |\n| ----------------------------- | -- |\n| Budget Theme                  | 1  |\n| Agriculture as the 1ˢᵗ engine | 3  |\n| MSMEs as the 2ⁿᵈ engine       | 6  |\n| Investment as the 3ʳᵈ engine  | 8  |\n| A. Investing in People        | 8  |\n| B. Investing in the Economy   | 10 |\n| C. Investing in Innovation    | 14 |\n| Exports as the 4ᵗʰ engine     | 15 |\n| Reforms as the Fuel           | 16 |\n| Fiscal Policy                 | 18 |\n\n# PART – B\n\n| Indirect taxes     | 20 |\n| ------------------ | -- |\n| Direct Taxes       | 23 |\n| Annexure to Part-A | 29 |\n| Annexure to Part-B | 31 |\n\n---\nNO_CONTENT_HERE\n---\n# Budget 2025-2026\n\n# Speech of Nirmala Sitharaman\n\n# Minister of Finance\n\n# February 1, 2025\n\nHon’ble Speaker,\n\nI present the Budget for 20